In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\HP\Downloads\code\recipe-recommender\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 768.57it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
emb1 = model.encode("rice")
emb2 = model.encode("basmati rice")

from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity([emb1], [emb2])

array([[0.7435677]], dtype=float32)

In [3]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def semantic_match_score(recipe_ings, user_ings, model, threshold=0.6):
    """
    Counts how many recipe ingredients are semantically
    similar to user ingredients using SBERT.
    """

    if not recipe_ings or not user_ings:
        return 0

    recipe_emb = model.encode(recipe_ings)
    user_emb = model.encode(user_ings)

    sim_matrix = cosine_similarity(recipe_emb, user_emb)

    # count recipe ingredients that match any user ingredient
    matches = (sim_matrix.max(axis=1) >= threshold).sum()

    return int(matches)

In [4]:
sample_recipe = ["basmati rice", "salt", "water"]
user_ings = ["rice", "salt"]

semantic_match_score(sample_recipe, user_ings, model)

2

In [5]:
import pandas as pd
import numpy as np

# load cleaned data from previous notebook
recipes = pd.read_csv("../data/recipes.csv")

# ⚠️ we must recreate clean_ingredients here OR save earlier dataframe

In [6]:
recipes = pd.read_pickle("../data/recipes_clean.pkl")

In [7]:
all_ingredients = set()

for ings in recipes["clean_ingredients"]:
    all_ingredients.update(ings)

unique_ingredients = sorted(list(all_ingredients))

len(unique_ingredients)

7310

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pickle

model = SentenceTransformer("all-MiniLM-L6-v2")

ingredient_list = sorted(list(all_ingredients))

ingredient_embeddings = model.encode(
    ingredient_list,
    batch_size=64,
    show_progress_bar=True
)

ingredient_embeddings.shape

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 679.98it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 115/115 [00:13<00:00,  8.75it/s]


(7310, 384)

In [9]:
with open("../data/ingredient_embeddings.pkl", "wb") as f:
    pickle.dump(
        {
            "ingredients": ingredient_list,
            "embeddings": ingredient_embeddings
        },
        f
    )

In [10]:
with open("../data/ingredient_embeddings.pkl", "rb") as f:
    data = pickle.load(f)

len(data["ingredients"]), data["embeddings"].shape

(7310, (7310, 384))

In [11]:
import pickle
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

with open("../data/ingredient_embeddings.pkl", "rb") as f:
    data = pickle.load(f)

ingredient_list = data["ingredients"]
ingredient_embeddings = data["embeddings"]

In [12]:
ingredient_to_index = {ing: i for i, ing in enumerate(ingredient_list)}

def get_embedding(ingredient):
    idx = ingredient_to_index.get(ingredient)
    if idx is None:
        return None
    return ingredient_embeddings[idx]

In [13]:
def semantic_match_fast(recipe_ings, user_ings, threshold=0.6):
    recipe_vecs = [get_embedding(i) for i in recipe_ings if get_embedding(i) is not None]
    user_vecs = [get_embedding(i) for i in user_ings if get_embedding(i) is not None]

    if not recipe_vecs or not user_vecs:
        return 0

    sim = cosine_similarity(recipe_vecs, user_vecs)

    matches = (sim.max(axis=1) >= threshold).sum()
    return int(matches)

In [14]:
sample_recipe = ["basmati rice", "salt", "water"]
user_ings = ["rice", "salt"]

semantic_match_fast(sample_recipe, user_ings)

2

In [16]:
import pandas as pd

candidate_recipes = pd.read_pickle("../data/candidate_recipes.pkl")


In [17]:
user_ingredients = ["onion", "salt", "rice"]

def compute_final_score(row):
    semantic = semantic_match_fast(row["clean_ingredients"], user_ingredients)
    return semantic + row["have_count"] - row["missing_cost"]

candidate_recipes["semantic_score"] = candidate_recipes["clean_ingredients"].apply(
    lambda x: semantic_match_fast(x, user_ingredients)
)

candidate_recipes["hybrid_score"] = (
    candidate_recipes["semantic_score"]
    + candidate_recipes["have_count"]
    - candidate_recipes["missing_cost"]
)

In [19]:
user_ingredients = ["onion", "salt", "rice"]

candidate_recipes["semantic_score"] = candidate_recipes["clean_ingredients"].apply(
    lambda x: semantic_match_fast(x, user_ingredients)
)

candidate_recipes["hybrid_score"] = (
    candidate_recipes["semantic_score"]
    + candidate_recipes["have_count"]
    - candidate_recipes["missing_cost"]
)

final_top = candidate_recipes.sort_values(
    by="hybrid_score", ascending=False
).head(10)

final_top[[
    "Name",
    "semantic_score",
    "have_count",
    "missing_count",
    "missing_cost",
    "hybrid_score"
]]

,Name,semantic_score,have_count,missing_count,missing_cost,hybrid_score
514698,White Rice,3,3,0,0,6
138984,Egyptian Rice for Fish,3,3,1,2,4
363941,Colombian Rice,3,3,1,2,4
129792,Pea Soup,2,2,0,0,4
231511,Guam Red Rice,3,3,1,2,4
52094,Neer Dosa,2,2,0,0,4
86331,Rice - Staple Dish of India,6,2,2,4,4
269477,Ricki Carroll’s 30-Minute Mozzarella - Homemade,2,1,0,0,3
120286,Sanna - Goan Rice Cakes,2,2,1,1,3
63080,Sauteed Bean Sprouts,2,2,1,2,2


In [20]:
final_data = candidate_recipes[[
    "RecipeId",
    "Name",
    "clean_ingredients",
    "missing_ings",
    "RecipeInstructions",
    "have_count",
    "missing_count",
    "missing_cost"
]]

final_data.to_pickle("../data/final_recommender_data.pkl")